# Part 2: ResNet-18 Model Improvement Ablations

This notebook keeps the Part 2 workflow notebook-owned while moving reusable training and result logic into project modules. The regular-training baseline is loaded from Part 1 results; the remaining ablations are trained here.

In [ ]:
from pathlib import Path
import importlib
import os
import sys


In [ ]:
_BOOTSTRAP_ROOT = None
_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / 'src').is_dir() and (_candidate / 'requirements.txt').exists():
        _BOOTSTRAP_ROOT = _candidate
        break

if _BOOTSTRAP_ROOT is None:
    _drive_root = Path('/content/drive/MyDrive/MLDS_Final_Project')
    if _drive_root.exists():
        _BOOTSTRAP_ROOT = _drive_root

if _BOOTSTRAP_ROOT is not None and str(_BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(_BOOTSTRAP_ROOT))

from src.utils.colab import bootstrap_notebook_runtime  # noqa: E402

ROOT = bootstrap_notebook_runtime(_BOOTSTRAP_ROOT)
ROOT


In [ ]:
import pandas as pd
from IPython.core.display import Image
from IPython.display import display

import src.evaluation.experiment_results as experiment_results

experiment_results = importlib.reload(experiment_results)
get_device = experiment_results.get_device
from src.utils.colab import print_colab_runtime_diagnostics, warn_if_colab_runtime_without_cuda  # noqa: E402
from src.utils.reproducibility import seed_everything  # noqa: E402


In [ ]:
import src.experiments.part2 as part2_experiments

part2_experiments = importlib.reload(part2_experiments)
experiment_output_paths = experiment_results.experiment_output_paths
load_part1_model_baseline_aggregated = experiment_results.load_part1_model_baseline_aggregated
run_part2_improvement_experiments = part2_experiments.run_part2_improvement_experiments
from src.utils.config import Part2ExperimentConfig  # noqa: E402


## Configuration

In [ ]:
config = Part2ExperimentConfig()
device = get_device(config)
warn_if_colab_runtime_without_cuda(device)
print_colab_runtime_diagnostics(device)
output_paths = experiment_output_paths(config.results_dir, config.figures_dir, config.part)
seed_everything(config.seed, deterministic=config.deterministic)

pd.DataFrame(
    [
        {
            'part': config.part,
            'config_name': config.config_name,
            'model_name': config.model_name,
            'tiles_per_side_values': config.tiles_per_side_values,
            'num_tile_permutations': config.num_tile_permutations,
            'epochs': config.epochs,
            'batch_size': config.batch_size,
            'num_workers': config.num_workers,
            'use_amp': config.use_amp,
            'device': str(device),
        }
    ]
)


In [ ]:
pd.DataFrame(config.ablations)

## Part 1 ResNet-18 Baseline

In [ ]:
part1_baseline_aggregated = load_part1_model_baseline_aggregated(config, config.model_name)
if not part1_baseline_aggregated.empty:
    display(part1_baseline_aggregated)

## Train Improvement Ablations

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    part2_results = run_part2_improvement_experiments(config=config, device=device)
    display(part2_results)
else:
    print('Training is skipped. Set RUN_TRAINING = True to train Part 2 ablations in this notebook.')

## Results

In [ ]:
results_path = Path(output_paths['aggregated_results'])

if results_path.exists():
    part2_results = pd.read_csv(results_path)
    if 'regular_part1' not in set(part2_results.get('ablation_name', [])) and not part1_baseline_aggregated.empty:
        part2_results = pd.concat([part1_baseline_aggregated, part2_results], ignore_index=True, sort=False)
    display(part2_results.sort_values(['tiles_per_side', 'ablation_name']))
else:
    print('Part 2 aggregated results were not found. Set RUN_TRAINING = True and run the training cell.')
    if part1_baseline_aggregated.empty:
        print('Part 1 ResNet-18 baseline results were also not found; run Part 1 first to include regular_part1.')

In [ ]:
figure_path = Path(output_paths['figure'])
if figure_path.exists():
    display(Image(filename=str(figure_path)))
else:
    print('Part 2 ablation figure has not been generated yet.')

In [ ]:
# Export this saved notebook to PDF. Save the notebook before running this cell,
# because nbconvert reads the on-disk .ipynb file rather than unsaved editor state.
import subprocess

notebook_path = ROOT / 'src' / 'notebooks' / 'part2_solution.ipynb'
export_dir = ROOT / 'outputs' / 'notebooks'
export_dir.mkdir(parents=True, exist_ok=True)

try:
    subprocess.run(
        [
            sys.executable,
            '-m',
            'jupyter',
            'nbconvert',
            '--to',
            'pdf',
            str(notebook_path),
            '--output-dir',
            str(export_dir),
        ],
        check=True,
    )
    print(f'Generated PDF: {export_dir / notebook_path.with_suffix(".pdf").name}')
except subprocess.CalledProcessError as exc:
    print('PDF export failed. Confirm the notebook is saved and that nbconvert plus LaTeX are installed.')
    print(f'Command exited with status {exc.returncode}.')
